# Manual Inspection of LLM Relevance Labels

This notebook is a lightweight manual audit tool for the LLM-generated relevance labels. It lets us look directly at randomly sampled queries and their judged candidate recipes instead of relying only on agreement metrics or retrieval metrics.

The notebook does not modify the ground truth files. It only loads:

- `groundtruth_outputs/annotation/llm_groundtruth_labels.jsonl`
- `groundtruth_outputs/annotation/blinded_annotation_items.jsonl`

and joins them into human-readable inspection tables.


## 1. Setup

Set the random seed and display size here. `NUM_RANDOM_QUERIES` controls how many queries are sampled for visual inspection. Each sampled query can show all 50 judged candidates, or fewer if you lower `MAX_DOCS_PER_QUERY_TO_DISPLAY`.


In [1]:
from __future__ import annotations

import json
from collections import Counter
from pathlib import Path
from typing import Optional

import pandas as pd
from IPython.display import display


def find_finalproject_root(start_path: Optional[Path] = None) -> Path:
    """Find the Finalproject directory by walking upward from the current path."""
    current_path = (start_path or Path.cwd()).resolve()
    for candidate_path in [current_path, *current_path.parents]:
        if candidate_path.name == "Finalproject":
            return candidate_path
        nested_finalproject = candidate_path / "Finalproject"
        if nested_finalproject.exists() and nested_finalproject.is_dir():
            return nested_finalproject.resolve()
    raise FileNotFoundError("Could not find the Finalproject directory.")


FINALPROJECT_ROOT = find_finalproject_root()
NOTEBOOK_OUTPUT_DIR = FINALPROJECT_ROOT / "notebooks" / "rec_and_eval" / "groundtruth_outputs"
ANNOTATION_OUTPUT_DIR = NOTEBOOK_OUTPUT_DIR / "annotation"

LLM_LABELS_PATH = ANNOTATION_OUTPUT_DIR / "llm_groundtruth_labels.jsonl"
BLINDED_ITEMS_PATH = ANNOTATION_OUTPUT_DIR / "blinded_annotation_items.jsonl"

RANDOM_SEED = 20260714
NUM_RANDOM_QUERIES = 8
MAX_DOCS_PER_QUERY_TO_DISPLAY = 50

# Options: "blinded_position" or "relevance_desc"
SORT_WITHIN_QUERY = "blinded_position"

RELEVANCE_LABEL_NAMES = {
    0: "0 - not relevant",
    1: "1 - partially related",
    2: "2 - relevant",
    3: "3 - highly relevant",
}

pd.set_option("display.max_colwidth", 220)
pd.set_option("display.max_rows", 80)

print("Finalproject root:", FINALPROJECT_ROOT)
print("LLM labels path:", LLM_LABELS_PATH)
print("Blinded items path:", BLINDED_ITEMS_PATH)


Finalproject root: C:\Users\tientm1\DS300-UIT-RecommenderSystem\Finalproject
LLM labels path: C:\Users\tientm1\DS300-UIT-RecommenderSystem\Finalproject\notebooks\rec_and_eval\groundtruth_outputs\annotation\llm_groundtruth_labels.jsonl
Blinded items path: C:\Users\tientm1\DS300-UIT-RecommenderSystem\Finalproject\notebooks\rec_and_eval\groundtruth_outputs\annotation\blinded_annotation_items.jsonl


## 2. Load Labels and Candidate Metadata

The label file stores only the LLM relevance score and stable identifiers. The blinded annotation item file stores the query text and recipe fields that were shown to the LLM. We join the two files by `(query_id, doc_id, blinded_position)`.


In [2]:
def load_jsonl_records(jsonl_path: Path) -> list[dict]:
    """Load a JSONL file into dictionaries."""
    records = []
    with jsonl_path.open("r", encoding="utf-8") as input_file:
        for line_number, line in enumerate(input_file, start=1):
            if not line.strip():
                continue
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError as error:
                raise ValueError(f"Invalid JSON at {jsonl_path}:{line_number}") from error
    return records


label_records = load_jsonl_records(LLM_LABELS_PATH)
blinded_item_records = load_jsonl_records(BLINDED_ITEMS_PATH)

labels_dataframe = pd.DataFrame(label_records)
items_dataframe = pd.DataFrame(blinded_item_records)

print("LLM label rows:", len(labels_dataframe))
print("Blinded item rows:", len(items_dataframe))
print("LLM label queries:", labels_dataframe["query_id"].nunique())
print("Blinded item queries:", items_dataframe["query_id"].nunique())

display(labels_dataframe.head(3))
display(items_dataframe.head(3))


LLM label rows: 25000
Blinded item rows: 25000
LLM label queries: 500
Blinded item queries: 500


,query_id,candidate_id,doc_id,blinded_position,relevance,label_match_strategy,label_reconciled,judge_type,llm_provider,llm_model_name,prompt_version
0,0,D001,3056,1,1,candidate_id,False,llm,openai_compatible_litellm,local-std-03,recipe_relevance_query_batch_title_type_description_v3
1,0,D002,2621,2,2,candidate_id,False,llm,openai_compatible_litellm,local-std-03,recipe_relevance_query_batch_title_type_description_v3
2,0,D003,2683,3,0,candidate_id,False,llm,openai_compatible_litellm,local-std-03,recipe_relevance_query_batch_title_type_description_v3


,query_id,doc_id,blinded_position,query_text,recipe_title,recipe_type,recipe_description,ingredients,normalized_ingredients,cooking_steps
0,0,3056,1,Bánh trung thu nướng nhân đậu xanh bằng nồi chiên không dầu,Bánh trung thu dẻo chay không dùng lò nướng,Món bánh,"Những chiếc bánh trung thu dẻo mềm, ngọt bùi thật không thể thiếu trong mùa trung thu. Vậy bạn an chay thì sao? Cùng Điện máy XANH vào bếp làm bánh dẻo trung thu chay đơn giản, nhanh chóng mà không cần dùng lò nướng .",250 g Đậu xanh đã ngâm qua đêm | 620 g Đường | 1/2 quả Nước chanh | 1/2 muỗng cà phê Muối | 40 ml Dầu ăn | 1 muỗng cà phê Nước hoa bưởi | 200 g Bột bánh dẻo,muỗng cà phê muối | đường | dầu ăn | bột bánh dẻo | đậu xanh đã ngâm qua đêm | nước hoa bưởi | quả nước chanh,"Bước 1: Làm nước đường: Cho 350 ml nước cùng 0.5 kg đường thêm ½ nước quả chanh để không bị lại đường rồi bắt lên bếp đun sôi, không khuấy khi nấu mà để cho đường tan hết. Sau khi nước đường sôi, để lửa nhỏ và nấu..."
1,0,2621,2,Bánh trung thu nướng nhân đậu xanh bằng nồi chiên không dầu,Bánh trung thu đậu xanh trứng muối sầu riêng với nồi cơm điện,Món bánh,"Bánh trung thu đậu xanh sầu riêng có vỏ bánh vàng óng, mềm tan, nhân bên trong thì ngọt bùi, thơm phức mùi sầu riêng. Đặc biệt, công thức mà Điện máy XANH chia sẻ ngay sau đây được làm hoàn toàn bằng nồi cơm điện. Và...",250 g Bột mì đa dụng (Phần vỏ bánh) | 2 cái Lòng đỏ trứng gà (Phần vỏ bánh) | 40 ml Dầu ăn (Phần vỏ bánh) | 20 g Bơ đậu phộng (Phần vỏ bánh) | 150 g Đậu xanh (Phần nhân bánh) | 50 g Sầu riêng (Phần nhân bánh) | 1 muỗ...,nước đường bánh nướng mua sẵn hoặc tự nấu | lòng đỏ trứng gà phần nước quết bánh | dầu ăn phần nhân bánh | đậu xanh phần nhân bánh | trứng vịt muối phần nhân bánh | lòng đỏ trứng gà phần vỏ bánh | đường phần nhân bán...,"Bước 1: Hầm đậu: Rửa sạch đậu xanh, cho vào nồi cùng 300ml nước và nấu khoảng 20 phút cho đậu chín nhừ. | Bước 2: Xay nhuyễn đậu và sầu riêng: Khi đậu vừa chín nhừ, bạn cho vào khoảng ¼ muỗng cà phê muối, 4 muỗng cà ..."
2,0,2683,3,Bánh trung thu nướng nhân đậu xanh bằng nồi chiên không dầu,Cách chiên bánh bao bằng nồi chiên không dầu vàng ruộm nhanh chóng tại nhà,Món bánh,"Thay vì phải chiên ngập dầu, bạn hoàn toàn có thể sử dụng nồi chiên không dầu để làm bánh bao chiên một cách nhanh chóng, ít dầu mỡ mà vẫn giữ được độ giòn ngon. Điện máy XANH sẽ cùng bạn vào bếp khám phá cách chiên ...",10 cái Bánh bao không nhân | 1 ít Dầu ăn (hoặc bơ) | 1 ít Sữa đặc,bánh bao không nhân | dầu ăn hoặc bơ | sữa đặc,"Bước 1: Chiên bánh bao: Bạn xếp bánh bao vào khay chiên, có thể lót thêm 1 lớp giấy nến để tăng khả năng chống dính. Tiếp đến, bạn dùng cọ quét một lớp dầu ăn mỏng lên trên các mặt bánh bao. Bạn đặt khay chiên vào nồ..."


## 3. Validate Coverage and Build the Inspection Table

Before manual inspection, this section checks that both files cover exactly the same 500 queries and 25,000 `(query, document)` pairs. Any mismatch should be fixed before trusting the visual inspection table.


In [3]:
EXPECTED_NUM_QUERIES = 500
EXPECTED_LABELS_PER_QUERY = 50
EXPECTED_TOTAL_LABELS = EXPECTED_NUM_QUERIES * EXPECTED_LABELS_PER_QUERY

required_label_columns = {"query_id", "doc_id", "blinded_position", "candidate_id", "relevance"}
required_item_columns = {
    "query_id",
    "doc_id",
    "blinded_position",
    "query_text",
    "recipe_title",
    "recipe_type",
    "recipe_description",
}

missing_label_columns = sorted(required_label_columns - set(labels_dataframe.columns))
missing_item_columns = sorted(required_item_columns - set(items_dataframe.columns))
if missing_label_columns:
    raise ValueError(f"Missing label columns: {missing_label_columns}")
if missing_item_columns:
    raise ValueError(f"Missing blinded item columns: {missing_item_columns}")

if len(labels_dataframe) != EXPECTED_TOTAL_LABELS:
    raise ValueError(f"Expected {EXPECTED_TOTAL_LABELS} label rows, found {len(labels_dataframe)}.")
if len(items_dataframe) != EXPECTED_TOTAL_LABELS:
    raise ValueError(f"Expected {EXPECTED_TOTAL_LABELS} blinded item rows, found {len(items_dataframe)}.")

labels_per_query = Counter(labels_dataframe.groupby("query_id").size())
items_per_query = Counter(items_dataframe.groupby("query_id").size())
if dict(labels_per_query) != {EXPECTED_LABELS_PER_QUERY: EXPECTED_NUM_QUERIES}:
    raise ValueError(f"Unexpected labels per query distribution: {dict(labels_per_query)}")
if dict(items_per_query) != {EXPECTED_LABELS_PER_QUERY: EXPECTED_NUM_QUERIES}:
    raise ValueError(f"Unexpected blinded items per query distribution: {dict(items_per_query)}")

if not labels_dataframe["relevance"].isin([0, 1, 2, 3]).all():
    raise ValueError("Found relevance labels outside the allowed 0-3 range.")

inspection_dataframe = labels_dataframe.merge(
    items_dataframe,
    on=["query_id", "doc_id", "blinded_position"],
    how="inner",
    validate="one_to_one",
)

if len(inspection_dataframe) != EXPECTED_TOTAL_LABELS:
    raise ValueError(
        f"Expected merged inspection table to have {EXPECTED_TOTAL_LABELS} rows, "
        f"found {len(inspection_dataframe)}."
    )

inspection_dataframe["relevance_name"] = inspection_dataframe["relevance"].map(RELEVANCE_LABEL_NAMES)
inspection_dataframe = inspection_dataframe.sort_values(["query_id", "blinded_position"]).reset_index(drop=True)

print("Coverage validation: OK")
print("Merged inspection rows:", len(inspection_dataframe))
print("Relevance distribution:")
display(
    inspection_dataframe["relevance"]
    .value_counts()
    .sort_index()
    .rename_axis("relevance")
    .reset_index(name="num_pairs")
)


Coverage validation: OK
Merged inspection rows: 25000
Relevance distribution:


,relevance,num_pairs
0,0,13505
1,1,7713
2,2,3013
3,3,769


## 4. Random Query-Level Inspection

This is the main manual check. For each sampled query, the notebook prints the query text, shows the relevance distribution within its 50 candidates, and displays the candidate recipes with the LLM relevance score.

Use `SORT_WITHIN_QUERY = "blinded_position"` to see the same order that was sent to the LLM. Use `SORT_WITHIN_QUERY = "relevance_desc"` to group highly relevant candidates at the top.


In [4]:
INSPECTION_COLUMNS = [
    "candidate_id",
    "blinded_position",
    "doc_id",
    "relevance",
    "relevance_name",
    "recipe_title",
    "recipe_type",
    "recipe_description",
    "normalized_ingredients",
]


def get_query_inspection_table(query_id: int, sort_within_query: str = SORT_WITHIN_QUERY) -> pd.DataFrame:
    """Return a readable inspection table for one query."""
    query_rows = inspection_dataframe[inspection_dataframe["query_id"] == query_id].copy()
    if query_rows.empty:
        raise ValueError(f"Unknown query_id={query_id}")

    if sort_within_query == "blinded_position":
        query_rows = query_rows.sort_values("blinded_position")
    elif sort_within_query == "relevance_desc":
        query_rows = query_rows.sort_values(["relevance", "blinded_position"], ascending=[False, True])
    else:
        raise ValueError("sort_within_query must be either 'blinded_position' or 'relevance_desc'.")

    available_columns = [column for column in INSPECTION_COLUMNS if column in query_rows.columns]
    return query_rows[available_columns].head(MAX_DOCS_PER_QUERY_TO_DISPLAY).reset_index(drop=True)


def display_query_inspection(query_id: int, sort_within_query: str = SORT_WITHIN_QUERY) -> None:
    """Print one query and display its candidate labels."""
    query_rows = inspection_dataframe[inspection_dataframe["query_id"] == query_id]
    query_text = query_rows["query_text"].iloc[0]
    relevance_distribution = query_rows["relevance"].value_counts().sort_index()

    print("=" * 120)
    print(f"query_id: {query_id}")
    print(f"query_text: {query_text}")
    print("relevance distribution:", dict(relevance_distribution))
    display(get_query_inspection_table(query_id, sort_within_query=sort_within_query))


sampled_query_ids = (
    inspection_dataframe[["query_id"]]
    .drop_duplicates()
    .sample(n=NUM_RANDOM_QUERIES, random_state=RANDOM_SEED)["query_id"]
    .tolist()
)

print("Sampled query IDs:", sampled_query_ids)
for sampled_query_id in sampled_query_ids:
    display_query_inspection(sampled_query_id, sort_within_query=SORT_WITHIN_QUERY)


Sampled query IDs: [242, 270, 485, 135, 151, 317, 360, 214]
query_id: 242
query_text: Bánh đa cá rô ăn kèm với rau cải cực ngon đơn giản ngay tại nhà
relevance distribution: {0: np.int64(15), 1: np.int64(27), 2: np.int64(7), 3: np.int64(1)}


,candidate_id,blinded_position,doc_id,relevance,relevance_name,recipe_title,recipe_type,recipe_description,normalized_ingredients
0,D001,1,4462,3,3 - highly relevant,Bánh đa cá rô ăn kèm với rau cải cực ngon đơn giản ngay tại nhà,Món nước,"Bánh đa cá rô có lẽ không quá xa lạ đối với chúng ta, tuy nguyên liệu dân dã dễ tìm nhưng mùi vị thanh ngọt của món nước này lại là thứ khiến ta khó quên. Hôm nay, hãy cùng vào bếp với Điện máy XANH để xem cách nấu b...",bó cải xanh | gừng | dầu ăn | bột nghệ | hành tím | gia vị thông dụng muối/ hạt nêm/ đường/tiêu | rau thì là | cá rô | hành lá | ngò | bánh đa
1,D002,2,6636,1,1 - partially related,Cơm cá rô giòn thơm hấp dẫn chuẩn vị miền Bắc,Món chiên,"Cá rô là một loại cá dân dã gắn với bữa cơm hằng ngày của người Việt Nam. Ngoài các cách nấu thường ngày, hôm nay Điện máy XANH xin giới thiệu với các bạn món chiên mới lạ của người miền Bắc - Cơm cá rô. Hãy cùng vào...",gừng | dầu ăn | hành tím | gạo | cá rô khoảng | rau thì là | hành phi | muối hạt | nước mắm | gia vị thông dụng hạt nêm/ muối/ tiêu
2,D003,3,7017,1,1 - partially related,"Cá rô chiên nước mắm cay nồng, thấm vị chuẩn cơm gia đình",Món chiên,"Cá rô chiên nước mắm là món chiên thơm ngon, hấp dẫn với cách làm siêu nhanh, đơn giản, thích hợp để bạn trổ tài nấu cho cả nhà thưởng thức vào những ngày bận rộn với công việc. Cùng vào bếp và xem ngay cách thực hiệ...",cá rô phi | dầu ăn | rượu trắng | gia vị thông dụng đường/ bột ngọt/ muối | nước mắm | ớt | tỏi
3,D004,4,6673,1,1 - partially related,Cá rô bí chiên giòn chấm nước mắm chua ngọt bằng chảo inox,Món chiên,"Cá rô bí hay cá rô non có thể chế biến thành nhiều món, trong đó có cá rô bí chiên giòn. Món chiên này không chỉ đơn giản, mà còn dễ ăn vì có thể ăn luôn cả xương cá. Vào bếp tìm hiểu ngay cách làm món ăn hấp dẫn này...",đường | dầu ăn | ớt băm | ói bột tẩm khô chiên giòn | sả băm | cá rô bí | nước mắm | nước cốt me | tỏi băm
4,D005,5,1597,1,1 - partially related,"Canh cá khoai rau cải ngọt ngon, dinh dưỡng cho cả nhà",Món canh,"Nếu như vô tình có 1 ngày bạn quá bận rộn không thể chu toàn được cho bữa cơm thì từ giờ đừng lo lắng nữa nhé, bởi đã có sự góp mặt của món canh nóng hổi, hấp dẫn, bổ dưỡng khiến ai cũng mê mẩn. Nào cùng bắt tay vào ...",rau cải ngọt | gia vị thông dụng hạt nêm/ muối/ đường/ tiêu xay | dầu ăn | gừng nhỏ | muỗng canh nước mắm | cá khoai | hành lá | tỏi
5,D006,6,6372,1,1 - partially related,"Cá rô kho củ cải trắng đậm đà, lạ vị đưa cơm tại nhà",Món kho,"Cá rô là nguyên liệu khá phổ biến trong bữa cơm hàng ngày ở mỗi gia đình với nhiều công thức khác nhau. Hôm nay, hãy cùng vào bếp với Điện máy XANH và thử làm ngay cá rô kho củ cải trắng, một món kho đơn giản nhưng l...",nước màu | dầu ăn | cá rô khoảng | củ cải trắng khoảng | nước mắm | hành lá | gia vị thông dụng đường/ tiêu/ bột ngọt/ hạt nêm | ớt | ngò rí
6,D007,7,7375,0,0 - not relevant,"Cá viên sốt cam lạ miệng, thơm ngon và bổ dưỡng",Món chiên,"Cá viên là một trong số những món ăn vặt hấp dẫn, đặc biệt đối với các bạn trẻ. Hôm nay, Điện máy XANH sẽ hướng dẫn cho các bạn làm cá viên sốt cam thơm ngon hấp dẫn ngay tại nhà. Cùng vào bếp thực hiện món chiên này...",nấm mèo | dầu ăn | cá rô phi | gia vị thông dụng đường/muối/tiêu/hạt nêm | bột ớt | cam | bột năng | hành phi
7,D008,8,6653,1,1 - partially related,"Cá rô chiên giòn sốt tỏi ớt đơn giản, thơm ngon, ăn là mê",Món chiên,"Công thức chi tiết cách làm cá rô chiên giòn sốt tỏi ớt đơn giản, thơm ngon, ăn là mê. Click xem và vào bếp làm ngay món ăn để cùng gia đình thưởng thức nhé!",gia vị muối/ giấm/ đường/ nước mắm/ tương ớt | cá rô khoảng | ớt sừng | tỏi
8,D009,9,1440,1,1 - partially related,Canh cá khoai cải cúc (tần ô) ngọt mát đưa cơm cho cả nhà,Món canh,"Như chúng ta thường thấy, một bữa cơm gia đình thì ít nhất phải có một món mặn và một món canh . Món canh thì rất đa dạng. Nhân đây, Điện máy XANH sẽ hướng dẫn bạn cách nấu canh cá khoai cải cúc vừa ngọt mát, vừa đơn...",cải cúc | dầu ăn | hành tím | cá khoai | nước lọc | gia vị thông dụ

query_id: 270
query_text: Bánh đúc lá lúa - bánh đúc đậu xanh nước cốt dừa mỡ hành thơm béo
relevance distribution: {0: np.int64(48), 1: np.int64(1), 3: np.int64(1)}


,candidate_id,blinded_position,doc_id,relevance,relevance_name,recipe_title,recipe_type,recipe_description,normalized_ingredients
0,D001,1,4514,0,0 - not relevant,"Kem đậu xanh sữa dừa thơm béo, mát lạnh dễ làm tại nhà",Món kem,"Kem đậu xanh là món kem rất được cả người lớn và trẻ em yêu thích bởi mùi vị ngọt ngào, béo bùi, thơm dịu và rất dễ ăn. Hôm nay, chuyên trang Vào bếp sẽ cùng bạn khám phá tuyệt chiêu chống nóng ngày hè với kem đậu xa...",đậu xanh nguyên vỏ | đường | sữa đặc | đậu xanh đãi vỏ | nước cốt dừa | sữa tươi có đường
1,D002,2,3668,1,1 - partially related,"Nước cốt dừa béo ngậy để chấm với bánh bò, bánh đúc ngọt,...",Món bánh,"Nước cốt dừa nấu sệt là một thành phần không thể thiếu trong nhiều món bánh , món chè truyền thống của người Việt Nam ta. Nếu bạn chưa biết cách nấu nước cốt dừa béo ngon thì mời bạn hãy vào bếp và thực hiện ngay côn...",đường | muối | nước lọc | nước cốt dừa đóng lon | bột năng | bột gạo
2,D003,3,2031,0,0 - not relevant,Bánh ú đậu trắng thơm bùi ngon dễ làm,Món bánh,"Bánh ú đậu trắng mềm dẻo, bùi thơm từ đậu, béo nhẹ vị nước cốt dừa. Với cách chế biến này, bạn có thể tùy ý dùng món bánh với bất kỳ món ăn kèm nào, từ đồ chua đến muối mè cũng đều rất ngon. Vào bếp cùng Điện máy XAN...",nếp | đậu trắng | muối | nước cốt dừa | lá chuối
3,D004,4,3492,0,0 - not relevant,Bánh ú nhân đậu xanh lá dứa thơm ngon hấp dẫn đơn giản,Món bánh,"Bánh ú (bánh bá trạng) là một loại bánh thường được làm vào dịp Tết Đoan Ngọ, được cả người Việt và người Hoa ưu chuộng trong dịp lễ này. Chuyên mục Vào bếp của Điện máy XANH hôm nay sẽ cùng bạn khám phá cách làm món...",hành tây băm nhỏ | đường | á lá dứa | dầu ăn | ói đường vani | cơm dừa bào sợi | gạo nếp | gia vị thông dụng đường/ muối/ tiêu | lá chuối dùng để gói bánh | đậu xanh đãi vỏ | nước cốt dừa | hạt nêm chay
4,D005,5,1957,0,0 - not relevant,Bánh gói lá chuối miền Trung thơm ngon béo ngậy hấp dẫn,Món bánh,"Bánh gói miền Trung với những nguyên liệu đơn giản như bột năng, bột gạo, nước cốt dừa,... lại làm say lòng biết bao thực khách nhờ hương vị thơm ngon, độc đáo. Các bạn hãy vào bếp cùng Điện máy XANH thực hiện món bá...",bột gạo | dầu ăn | đậu xanh | đầu hành cắt nhỏ | bột năng | nước cốt dừa | muối/đường | nước bột gạo | hành lá cắt nhỏ | mè rang
5,D006,6,2770,0,0 - not relevant,"Bánh cuốn ngọt miền Tây nhân đậu xanh dừa thơm ngon, hấp dẫn",Món bánh,"Bánh cuốn ngọt nhân đậu xanh dừa - một món đặc sản bạn nên nếm thử khi ghé thăm miền Tây sông nước. Món bánh có lớp vỏ dai mềm, thơm nhẹ mùi mè rang cùng phần nhân đậu xanh bùi ngọt, beo béo, vô cùng hấp dẫn. Nếu bạn...",muỗng cà phê muối | bột bánh cuốn pha sẵn | đường | dầu ăn | lá cẩm | dừa nạo sợi | đậu xanh | lá dứa | mè trắng | dừa nạo nhuyễn để lấy nước cốt
6,D007,7,4472,0,0 - not relevant,Kem đậu xanh nước cốt dừa bằng máy xay sinh tố dẻo mịn thơm ngon,Món kem,"Vào hè rồi bạn đã nghĩ ra món ăn vặt nào thanh mát, giải nhiệt chưa? Hôm nay, Điện máy XANH sẽ hướng dẫn bạn cách làm kem đậu xanh nước cốt dừa bằng máy xay sinh tố dẻo mịn thơm ngon. Cùng vào bếp thực hiện ngay món ...",bột bắp | sữa tươi không đường | sữa đặc | nước cốt dừa | đậu xanh cà vỏ
7,D008,8,3327,0,0 - not relevant,Bánh bò da lợn mới lạ thơm ngon hấp dẫn đơn giản,Món bánh,"Bánh bò da lợn - một sự kết hợp hoàn hảo giữa 2 món bánh dân dã đem đến một hương vị béo ngọt, thơm ngon đến khó cưỡng. Nếu bạn cũng muốn nếm thử món bánh này thì hãy vào bếp ngay cùng Điện máy XANH nhé!",đường | bột bánh bò | đậu xanh | lá dứa tươi | bột năng | nước cốt dừa | ói men khô có sẵn trong bột bánh bò | bột gạo | nước dừa tươi
8,D009,9,3441,0,0 - not relevant,Bánh ít trần khoai mì nhân đậu xanh thơm ngon dẻo mềm dễ làm,Món bánh,"Chỉ với vài nguyên liệu rẻ tiền từ khoai mì, đậu xanh, bạn đã có thể tạo nên một món ăn vặt cực kỳ thơm ngon, hấp dẫn cho gia đình cùng thưởng thức. Hãy vào bếp ngay cùng Điện máy XANH để học cách làm bánh ít trần kh...",đường | khoai mì | muối | đậu xanh | hạt mè trắng | dừa nạo xay | bột năng | nước cốt dừa | bó lá dứa | nư

query_id: 485
query_text: Bắp bò hầm bạch quả đậm đà bổ dưỡng
relevance distribution: {0: np.int64(5), 1: np.int64(38), 2: np.int64(5), 3: np.int64(2)}


...

,candidate_id,blinded_position,doc_id,relevance,relevance_name,recipe_title,recipe_type,recipe_description,normalized_ingredients
0,D001,1,9796,0,0 - not relevant,"Sữa đậu Hà Lan và sinh tố đậu Hà Lan thơm béo, đầy dinh dưỡng",Thức uống,"Sữa và sinh tố là một trong những thức uống dinh dưỡng, với nhiều tác dụng tốt đối với sức khỏe. Hôm nay, Điện máy XANH sẽ cùng vào bếp mách bạn 2 cách làm sữa đậu Hà Lan và sinh tố đậu Hà Lan thơm béo, đơn giản tại ...",sữa đặc on | nước | đậu hà lan
1,D002,2,4496,0,0 - not relevant,[Video] Cách làm kem chuối cacao bằng máy xay sinh tố đơn giản tại nhà,Món kem,Có phải bạn luôn nghĩ để có làm được một viên kem tươi mát phải cần có máy làm kem và rất nhiều nguyên liệu đắt tiền? Hãy tạm gác suy nghĩ đó qua một bên vì hôm nay chuyên mục Vào bếp sẽ hướng dẫn cho các bạn làm kem...,chuối lớn | hạt đậu phộng dùng để trang trí | bột ca cao | bơ đậu phộng | sữa tươi tùy theo sở thích
2,D003,3,9847,1,1 - partially related,Sinh tố bơ đậu phộng thơm béo cho bữa sáng đầy dinh dưỡng,Sinh tố,"Bổ sung dinh dưỡng mỗi sáng với món sinh tố bơ đậu phộng thơm ngon, bùi béo để cho cơ thể có thêm nhiều năng lượng cho ngày mới thật năng động nhé! Cùng vào bếp với Điện máy XANH xem ngay công thức và thực hiện món s...",muỗng cà phê muối | đậu phộng rang giã dập | whipping cream | bột bơ đậu phộng peanut butter powder | sữa hạnh nhân không đường hoặc sữa tươi không đường | bơ đậu phộng | topping cream dùng trang trí | đá viên | bột ...
3,D004,4,8946,0,0 - not relevant,"Đậu phộng nướng ngũ vị hương thơm ngon, giòn tan dễ làm",Ăn vặt,Đậu phộng nướng ngũ vị là một trong những món ăn vặt mà bạn không nên bỏ qua bởi cách làm khá đơn giản và hương vị lại vô cùng thơm ngon. Cùng bài vào bếp dưới đây thực hiện món ăn này nhé!,muỗng canh ngũ vị hương | nước cốt dừa | đậu phộng
4,D005,5,9014,0,0 - not relevant,Đậu phộng ngào đường giòn tan đơn giản nhâm nhi cả ngày,Ăn vặt,"Để các cuộc trò chuyện không còn nhàm chán hay là khi thưởng thức 1 bộ phim được gay cấn hơn thì không thể thiếu sự xuất hiện của các món ăn vặt rồi! Nhân đây, hôm nay Điện máy XANH sẽ tiết lộ cho mọi người món đậu p...",đường | muối | bột bắp | gừng nhỏ | đậu phộng
5,D006,6,2495,0,0 - not relevant,Kẹo socola đậu phộng giòn giòn cả nhà đều thích,Món bánh,"Kẹo socola đậu phộng giòn tan, thơm lừng lại có cách làm đơn giản đến bất ngờ. Các bạn cùng vào bếp với Điện máy XANH làm món ăn vặt hấp dẫn này nhé!",muối | socola đen 70 | socola sữa | đậu phộng
6,D007,7,6999,0,0 - not relevant,Đậu phụ từ sữa đậu nành nhanh gọn đơn giản tại nhà,Món chiên,"Đậu phụ làm từ sữa đậu nành nhanh gọn, đơn giản nhưng không kém phần thơm ngon, hấp dẫn. Món hấp này có lớp vỏ ngoài vàng giòn cùng với độ mềm mịn, béo ngậy bên trong thì chắc chắn sẽ đem đến cho bạn một trải nghiệm ...",muỗng cà phê muối | sữa đậu nành ít đường | dầu ăn | bột bắp | trứng gà
7,D008,8,9648,0,0 - not relevant,"Sữa đậu nành hột gà béo ngậy, hấp dẫn tại nhà",Thức uống,"Mách bạn cách làm sữa đậu nành hột gà thơm bùi, béo ngậy vừa ngon miệng lại vừa bổ dưỡng. Với cách chế biến đơn giản, kết hợp hài hòa giữa các nguyên liệu nên thức uống không hề có mùi tanh từ trứng. Hãy cùng Điện má...",sữa đậu nành | lòng đỏ trứng gà | bột cacao không bắt buộc | sữa đặc
8,D009,9,9741,0,0 - not relevant,Sữa đậu ngự mix hạt điều bằng máy nấu sữa hạt thơm ngon đơn giản,Thức uống,"Sữa hạt là một thức uống thơm ngon, bổ dưỡng và đặc biệt được nhiều chị em yêu thích vì có công dụng đẹp da, giữ dáng. Hôm nay, hãy vào bếp cùng Điện máy XANH thực hiện món sữa đậu ngự mix hạt điều cực ngon và đơn gi...",đường thốt nốt hoặc đường phèn | nước lọc | hạt điều | đậu ngự
9,D010,10,9730,0,0 - not relevant,Sữa đậu nành bằng máy làm sữa hạt thơm béo không bã,Thức uống,"Bạn muốn tự tay làm sữa đậu nành thơm ngon, sánh mịn ngay tại nhà? Với công thức cách làm sữa đậu nành bằng máy làm sữa hạt này, dù bạn là người mới Vào bếp cũng dễ dàng có ngay một thức uống bổ dưỡng, không cần lọc ...",mè đen | hạnh nhân đã sấy | gia vị thông dụng đường/đá | đậ

query_id: 360
query_text: Đậu phụ nhồi thịt sốt cà chua đơn giản cho bữa cơm hàng ngày
relevance distribution: {0: np.int64(23), 1: np.int64(18), 2: np.int64(8), 3: np.int64(1)}


,candidate_id,blinded_position,doc_id,relevance,relevance_name,recipe_title,recipe_type,recipe_description,normalized_ingredients
0,D001,1,171,0,0 - not relevant,Mỳ Ý Spaghetti thịt bò bằm sốt cà chua đơn giản tại nhà,Món ngon hàng ngày,"Tranh thủ cà chua đang vào mùa ngon và rẻ, bạn đổi vị cho các bé với món mỳ Ý thơm ngon như ngoài hàng.",2- tỏi | đường | bơ | hành tây | thịt bò xay | lá oregano | giấm | gia vị: muối | mỳ spaghetti | cà chua chín | tương cà chua | bột basil | hạt tiêu | dầu ô liu
1,D002,2,1191,1,1 - partially related,Đậu phụ Tứ Xuyên sốt cay,Món chay,Đậu phụ Tứ Xuyên cay này rất dễ.\r\nHôm nay nấu món này nha,gừng | sa tế | nấm hương | hành tím | đậu phụ | tỏi
2,D003,3,7403,1,1 - partially related,"Cà tím nhồi đậu phụ (đậu hũ) thơm ngon, đơn giản tại nhà",Món chiên,"Hôm nay Điện máy XANH sẽ hướng dẫn các bạn làm một món chiên từ cà tím và đậu phụ, đây là một món ăn đậm vị và vô cùng hấp dẫn. Thích hợp cho những ngày ăn chay. Cùng vào bếp ngay để thực hiện món chiên đơn giản mà l...",cà tím | nấm mèo | dầu ăn | gia vị thông dụng muối/ đường/ hạt nêm/ tiêu | đậu hũ trắng đậu phụ | nước tương | hành lá | ớt hiểm | tỏi băm | củ cà rốt | ngò rí
3,D004,4,1190,0,0 - not relevant,Bún xào chay cho ngày rằm,Món chay,Những ngày rằm thay vì những món ăn chay với cơm thì hôm nay làm món bún xào chay này sẽ tiết kiệm được thời gian cho bạn cũng đảm bảo được độ ngon mà nó mang lại.,bún gạo | đậu phụ chiên | cà rốt | cải thìa
4,D005,5,1411,0,0 - not relevant,Canh chua rau muống chay thanh đạm mà vẫn ngon cơm,Món canh,"Những món ăn từ thịt cá đôi khi khiến bạn ngán ngẩm và ăn cơm không ngon miệng, những lúc như thế hãy nghĩ đến món canh chua rau muống nấu chay, vừa thanh đạm vừa bổ dưỡng, cả nhà cùng tấm tắc khen ngon. Còn bây giờ ...",dầu ăn | nấm hải sản | đậu phụ chiên khoảng | rau muống | rau nêm ngổ/ ngò gai | gia vị thông dụng muối/ đường/ bột ngọt | sấu | ớt | cà chua | tỏi
5,D006,6,871,2,2 - relevant,Đậu hũ chiên sốt cà chua,Món chay,"Đậu hũ chiên sốt cà chua là món ăn không còn xa lạ với nhiều người, nhất là các bạn sinh viên. Món này hội tụ 3 yếu tố ngon, bổ, rẻ. Từng miếng đậu hũ mềm, thấm đậm gia vị, thêm phần nước sốt chua chua, ngọt ngọt thậ...",hành boa rô | dầu ăn | đường trắng cafe | muối cafe | đậu hũ chiên | bột ngọt | cà chua
6,D007,7,7437,0,0 - not relevant,"Trứng sốt cà chua đơn giản, ngon miệng cho bữa cơm nhà tròn vị",Món chiên,"Bạn đang tìm món trứng sốt cà chua ngon miệng, dễ làm cho bữa cơm gia đình? Vào bếp ngay để khám phá cách làm trứng gà sốt cà chua đơn giản, biến tấu bữa ăn thêm hấp dẫn với món chiên này nhé. Đảm bảo cả nhà sẽ thích...",hành tím | cà chua khoảng | nước lọc | gia vị thông dụng tương ớt/ dầu hào/ đường/ hạt nêm/ bột ngọt/ nước mắm/ dầu ăn/ muối | hành lá | trứng gà | ngò rí
7,D008,8,1428,0,0 - not relevant,Canh chua đậu chay đơn giản cho bữa cơm chay thêm vị,Món canh,Canh chua đậu chay là một trong những món chay hấp dẫn lại còn rất dẽ thực hiện. Bài viết vào bếp dưới đây sẽ hướng dẫn đến bạn cách thực hiện món canh chua thơm ngon này.,gia vị đường/hạt nêm chay/nước mắm chay | tắc | dầu ăn | các loại đậu đậu rồng/đậu bắp/đậu đũa | rau thơm ngò gai/rau ngò om | đậu hũ | thơm | ớt | cà chua
8,D009,9,4680,0,0 - not relevant,"Lasagna đơn giản, đúng chuẩn kiểu Ý ngay tại nhà",Món nướng,"Bạn đã bao giờ nghe đến món lasagna - một loại mì của nền ẩm thực Ý chưa? Hôm nay hãy để Điện máy XANH vào bếp cùng bạn thử sức thực hiện món ăn độc đáo này với cách làm đơn giản, dễ thực hiện mà lại đúng chuẩn này nhé!",cà chua nghiền đóng hộp | tỏi xay | thịt bò xay | lá mì lasana | trứng | hành tây cắt hạt lựu | sốt cà chua cô đặc tùy loại | phô mai ricotta | phô mai parmesan rắc lên bề mặt | sốt cà chua | dầu ô liu | phô mai mozz...
9,D010,10,857,1,1 - partially related,Rau cuộn đậu phụ,Món chay,"Từng miếng rau cuộn chỉ nhìn thôi cũng thấy hấp dẫn rồi. Đảm bảo khi ăn món này bạn sẽ thích ngay! Món ăn vừa ngọt lại béo ngậy hòa với sốt chua cay rất ngon. Đặc biệt, với món rau cuộn này dù bạn có ăn 

query_id: 214
query_text: Bánh sinh nhật rau câu trái cây dễ làm, không cần máy đánh trứng
relevance distribution: {0: np.int64(3), 1: np.int64(45), 3: np.int64(2)}


,candidate_id,blinded_position,doc_id,relevance,relevance_name,recipe_title,recipe_type,recipe_description,normalized_ingredients
0,D001,1,3250,1,1 - partially related,Bánh thạch dâu tây sữa chua thơm ngon đơn giản ai ai cũng ghiền,Món bánh,"Thạch rau câu là một trong những món tráng miệng yêu thích của trẻ nhỏ lẫn người lớn. Dưới đây, cùng vào bếp với Điện máy XANH thực hiện cách làm bánh thạch dâu tây sữa chua tươi mát, giàu vitamin đảm bảo ai ai cũng ...",đường | dâu tây | bột rau câu agar | phô mai | sữa chua | đường đen hoặc bột cacao/cà phê
1,D002,2,3021,3,3 - highly relevant,"Bánh sinh nhật rau câu trái cây dễ làm, không cần máy đánh trứng",Món bánh,"Bạn muốn tự tay làm một chiếc bánh sinh nhật độc đáo, mát lạnh và tốt cho sức khỏe mà không cần đến lò nướng hay máy đánh trứng phức tạp? Bánh sinh nhật rau câu trái cây chính là lựa chọn lý tưởng. Trong bài viết dướ...",thanh long ruột trắng | đường | nước cốt chanh | cream cheese kem phô mai | bơ nhạt | thanh long ruột đỏ | sữa chua không đường | bánh quy bơ | tinh chất vani | bột gelatin | heavy cream kem sữa béo
2,D003,3,679,0,0 - not relevant,Làm bánh lòng trắng trứng béo xốp không cần bột,"Món tráng miệng, giải khát","Chỉ cần 3 lòng trắng trứng gà và một muỗng đường, bạn có ngay chiếc bánh nướng thơm, vừa béo vừa xốp.",trứng gà:
3,D004,4,8859,1,1 - partially related,Rau câu trái cây thơm ngon thanh mát cho cả nhà,Ăn vặt,"Rau câu trái cây là món tráng miệng thơm ngon, thích hợp giải nhiệt ngày nắng nóng. Hôm nay, bạn hãy cùng chị Ngọc Hiền vào bếp thực hiện ngay món rau câu thanh mát, chuẩn vị này cho cả nhà cùng thưởng thức sau mỗi b...",đường | xoài chín cỡ vừa | bột rau câu dẻo | nước lọc | thanh long cỡ vừa
4,D005,5,9951,1,1 - partially related,"Thạch dưa lưới lạ vị, thơm mát, giải nhiệt",Món tráng miệng,"Đến mùa dưa lưới rồi, bạn đã nghĩ ra món tráng miệng nào được làm ra từ loại quả thanh mát này chưa? Hôm nay, Điện máy XANH sẽ mách bạn cách làm thạch dưa lưới lạ vị, thơm mát, giải nhiệt. Cùng vào bếp thực hiện ngay...",đường | bột rau câu dẻo | dưa lưới khoảng
5,D006,6,9992,1,1 - partially related,"Rau câu sữa milo nhanh chóng, ngon cực đỉnh chỉ với 10 phút",Món tráng miệng,"Chớ vội lầm tưởng sữa milo chỉ có thể là món thức uống bổ dưỡng mà thôi, đó là do bạn chưa biết đến món rau câu sữa milo hấp dẫn, thanh mát, hạ nhiệt vào các ngày nắng nóng cực hiệu quả. Nào hãy cùng chị Ngọc Hiền và...",đường | nước | sữa milo hộp | bột rau câu giòn
6,D007,7,9969,1,1 - partially related,Món trứng pha lê độc lạ bắt mắt cho bữa tráng miệng,Món tráng miệng,"Rau câu hình trái trứng trong veo, đẹp mắt cùng với hương vị thơm ngon sẽ làm cho bữa cơm gia đình bạn thú vị hơn rất nhiều. Hãy cùng vào bếp ngay với Điện máy XANH làm trứng pha lê - món tráng miệng vô cùng độc đáo,...",ói bột rau câu | gia vị thông dụng muối/ đường/ hạt nêm | hành lá | trứng gà | bông cải trắng | tôm tươi
7,D008,8,8954,1,1 - partially related,"Rau câu sợi giòn thơm béo, giòn sựt sựt lạ miệng với nồi inox",Ăn vặt,"Với hương vị ngọt thanh, giòn sựt sựt và chút béo thơm, rau câu sợi giòn là món tráng miệng vừa ngon miệng vừa đẹp mắt, thích hợp cho cả gia đình thưởng thức. Chỉ cần một chiếc nồi inox và vài nguyên liệu đơn giản, b...",đường | sữa đặc | bột rau câu giòn agar | ói cafe đen | rau câu sợi | nước cốt lá dứa xay từ lá dứa tươi và nước | nước cốt dừa
8,D009,9,9187,1,1 - partially related,Rau câu thanh long ruột đỏ đẹp mắt bằng máy xay sinh tố,Ăn vặt,"Mùa nắng nóng đã bắt đầu với nền nhiệt ngày càng cao. Để chăm sóc sức khoẻ, bạn hãy thử chế biến những món có công dụng giải nhiệt. Trong số đó, đừng quên món ăn vặt quen thuộc rau câu thanh long. Vào bếp trổ tài để ...",thanh long ruột đỏ | đường | nước | bột rau câu giòn
9,D010,10,3221,1,1 - partially related,Rau câu 4D hoa nổi hương bắp thơm ngon đẹp mắt đơn giản tại nhà,Món bánh,Rau câu là một món tráng miệng quen thuộc với mọi người. Hãy cùng vào bếp làm ngay món rau câu 4D hoa nổi vừa thơm ngon lại vừa đẹp mắt cho gia đình mình nhé!,sữa tươi | đườn

## 5. Targeted Spot Checks by Relevance Level

Random query inspection is useful, but it can miss edge cases. This section samples individual `(query, document)` pairs from each relevance level so we can quickly inspect whether labels `0`, `1`, `2`, and `3` feel calibrated.


In [5]:
TARGETED_SAMPLES_PER_RELEVANCE_LEVEL = 8

targeted_rows = []
for relevance_level in [3, 2, 1, 0]:
    level_rows = inspection_dataframe[inspection_dataframe["relevance"] == relevance_level]
    sample_size = min(TARGETED_SAMPLES_PER_RELEVANCE_LEVEL, len(level_rows))
    if sample_size == 0:
        continue
    targeted_rows.append(
        level_rows.sample(
            n=sample_size,
            random_state=RANDOM_SEED + relevance_level,
        )
    )

targeted_inspection_dataframe = (
    pd.concat(targeted_rows, ignore_index=True)
    .sort_values(["relevance", "query_id", "blinded_position"], ascending=[False, True, True])
    .reset_index(drop=True)
)

targeted_columns = [
    "query_id",
    "query_text",
    "candidate_id",
    "blinded_position",
    "doc_id",
    "relevance",
    "relevance_name",
    "recipe_title",
    "recipe_type",
    "recipe_description",
]

display(targeted_inspection_dataframe[targeted_columns])


,query_id,query_text,candidate_id,blinded_position,doc_id,relevance,relevance_name,recipe_title,recipe_type,recipe_description
0,128,"Chim cút xào lăn biến tấu lạ miệng, thơm ngon, ăn là ghiền",D017,17,5331,3,3 - highly relevant,"Chim cút xào lăn biến tấu lạ miệng, thơm ngon, ăn là ghiền",Món xào,Nếu thực đơn món xào của bạn đã nhàm chán với các món quá quen thuộc thì hôm nay hãy vào bếp ngay cùng Điện máy XANH để thực hiện món chim cút xào lăn lạ miệng thơm ngon với cách làm đơn giản vô cùng này nhé!
1,138,Món vả kho thịt cực đậm đà thơm ngon cho bữa ăn đưa cơm,D024,24,6423,3,3 - highly relevant,Thịt kho sung thơm ngon lạ miệng đậm đà cho bữa cơm,Món kho,"Những món kho luôn đem đến một hương vị đậm đà, vị mằn mặn ăn kèm chén cơm nóng mà nghĩ thôi cũng khiến bạn cồn cào cái bụng rồi. Cùng vào bếp ngay hôm nay thực hiện món thịt kho sung lạ miệng cho gia đình nhé!"
2,166,Bánh milo tan chảy không cần lò nướng hấp dẫn đơn giản tại nhà,D009,9,1976,3,3 - highly relevant,Bánh bông lan milo socola tan chảy mềm mịn không cần lò nướng,Món bánh,"Bánh bông lan milo được hòa quyện giữa lớp kem tươi béo ngậy cùng cốt bánh ngọt đắng, cực kỳ thơm ngon và ăn không có cảm giác ngán. Bạn cũng có thể thực hiện thành công món bánh này chỉ với một chiếc xửng hấp. Vào b..."
3,331,Công thức chi tiết cách tự làm dầu hào cực ngon đơn giản tại nhà,D026,26,10063,3,3 - highly relevant,Công thức chi tiết cách tự làm dầu hào cực ngon đơn giản tại nhà,Món khô - mắm,"Là một trong những gia vị cực kì phổ biến và được yêu thích trong ẩm thực, dầu hào hay còn gọi là dầu hàu đã và đang khiến nhiều người yêu thích vì hương vị của mình. Cùng Điện máy XANH vào bếp tìm hiểu công thức tự ..."
4,341,Nước chấm gỏi cuốn chay từ đậu phộng thơm ngon khó cưỡng tại nhà,D046,46,837,3,3 - highly relevant,Tương chấm gỏi cuốn chay,Món chay,"Tương chấm gỏi cuốn chay với vị béo ngậy của bơ đậu phộng, vị cay cay của ớt trái, đem đến cho bạn món nước chấm gỏi cuốn rất ngon và hấp dẫn đấy nhé!"
5,450,Bò bía mặn chấm tương đen đậu phộng béo ngậy ngon mê ly,D032,32,10242,3,3 - highly relevant,Bò bía mặn chấm tương đen đậu phộng béo ngậy ngon mê ly,Món cuốn - trộn,"Bò bía mặn là một món cuốn vô cùng ngon miệng với sự kết hợp độc đáo giữa các nguyên liệu như trứng chiên, lạp xưởng, tôm khô, rau quế,... Nếu bạn đã từng thưởng thức qua và cảm thấy nhung nhớ hương vị của cuốn bò bí..."
6,482,Bánh bò rễ tre bằng bột pha sẵn mềm dai chuẩn vị miền tây,D047,47,2385,3,3 - highly relevant,Bánh bò rễ tre bằng bột pha sẵn mềm dai chuẩn vị miền tây,Món bánh,"Bánh bò là món bánh được nhiều người yêu thích nhờ mùi vị thơm ngon, cốt bánh mềm mịn, dẻo thơm và màu sắc đẹp mắt. Hôm nay, cùng Điện máy XANH vào bếp học cách làm bánh bò rễ tre bằng bột pha sẵn mềm dai chuẩn vị mi..."
7,484,"Sườn xào chua ngọt mềm thơm ngon, đậm đà, cực hao cơm tại nhà",D008,8,953,3,3 - highly relevant,Sườn xào chua ngọt,Món chính,"Vị chua chua ngọt ngọt, miếng sườn mềm thơm hòa quyện cùng nước sốt sền sệt đậm đà, chỉ nhìn thôi đã thấy ngon miệng vô cùng."
8,39,Xôi lạc bằng nồi cơm điện cực nhanh gọn mà ngon,D040,40,338,2,2 - relevant,Xôi đậu đen cấp tốc bằng nồi cơm điện,Món ngon hàng ngày,"Hạt xôi dẻo mềm, đậu đen bên trong mềm bở mà vẫn giữ nguyên hạt, ăn kèm muối lạc làm nên bữa sáng vừa nhanh gọn lại thơm ngon."
9,93,"Bánh bao nhân custard xoài mềm mịn, thơm ngon",D036,36,3107,2,2 - relevant,Bánh bao nhân dừa ngọt căng mềm mịn thơm ngon,Món bánh,"Bánh bao nhân đậu đỏ chay thơm ngon, béo ngậy, là món ăn mà cả gia đình bạn có thể thưởng thức vào buổi sáng. Hãy cùng vào bếp thực hiện ngay món bánh ngon lành này nhé!"


## 6. Inspect a Specific Query

Use this cell when a query looks suspicious in the evaluation report or when you want to manually audit one known `query_id`. Set `MANUAL_QUERY_ID` to an integer from `0` to `499`.


In [ ]:
MANUAL_QUERY_ID = None
MANUAL_SORT_WITHIN_QUERY = "relevance_desc"

if MANUAL_QUERY_ID is None:
    print("Set MANUAL_QUERY_ID to an integer query_id when you want to inspect a specific query.")
else:
    display_query_inspection(int(MANUAL_QUERY_ID), sort_within_query=MANUAL_SORT_WITHIN_QUERY)


## 7. Optional Export of the Current Random Sample

This cell is disabled by default. Turn on `EXPORT_RANDOM_SAMPLE` if you want a CSV snapshot of the currently sampled query inspections for sharing or offline review.


In [ ]:
EXPORT_RANDOM_SAMPLE = False
RANDOM_SAMPLE_EXPORT_PATH = NOTEBOOK_OUTPUT_DIR / "manual_inspection" / "sampled_llm_relevance_inspection.csv"

if EXPORT_RANDOM_SAMPLE:
    RANDOM_SAMPLE_EXPORT_PATH.parent.mkdir(parents=True, exist_ok=True)
    sampled_export_dataframe = inspection_dataframe[inspection_dataframe["query_id"].isin(sampled_query_ids)].copy()
    sampled_export_dataframe = sampled_export_dataframe.sort_values(["query_id", "blinded_position"])
    sampled_export_dataframe.to_csv(RANDOM_SAMPLE_EXPORT_PATH, index=False, encoding="utf-8-sig")
    print("Saved random sample inspection file to:", RANDOM_SAMPLE_EXPORT_PATH)
else:
    print("Export disabled. Set EXPORT_RANDOM_SAMPLE = True to write a CSV snapshot.")
